# vLLM benchmark: Qwen2.5-1.5B-Instruct on a cloud GPU

Part of [serve_llm](https://github.com/<your-github-username>/serve_llm): the same model served with
llama.cpp on a laptop CPU and with vLLM here, measured with the same harness (`bench/run_bench.py`).

**Before running**
- **Kaggle:** Settings → Accelerator → **GPU T4 x2** (one GPU is used), and Settings → **Internet on**.
  vLLM does not support the P100 option.
- **Colab:** Runtime → Change runtime type → **T4 GPU**.
- Run the cells in order. Installing vLLM takes a few minutes; if the notebook asks to restart
  after the install, restart and continue from step 3.

**What it measures**
1. Concurrency sweep (1 → 32) at FP16: continuous batching under load.
2. Prefix caching on vs. off, with prompts that share a ~1.45k-token system prompt.
3. *(Optional)* the official AWQ 4-bit model, same sweep.

Results are written as JSON to `results/`; download them and commit them to the repo's `results/`.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install vLLM and httpx

Installing vLLM upgrades PyTorch, but leaves the image's preinstalled `torchaudio` built for
the old CUDA version. `transformers` imports torchaudio whenever it is present, and the
mismatch crashes `vllm serve` at startup. vLLM does not need torchaudio for a text model,
so it is removed.

In [ ]:
%pip install -q vllm httpx
%pip uninstall -y -q torchaudio

In [ ]:
!python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

## 3. Configuration

In [ ]:
import importlib.metadata
import json
import os
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
AWQ_MODEL = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"
SERVED_NAME = "qwen2.5-1.5b-instruct"  # same model name the laptop runs use
PORT = 8000
BASE_URL = f"http://127.0.0.1:{PORT}/v1"

GITHUB_USER = "<your-github-username>"  # used to download the harness; or upload the files
REPO_RAW = f"https://raw.githubusercontent.com/{GITHUB_USER}/serve_llm/main"

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
RESULTS = WORK / "results"
RESULTS.mkdir(exist_ok=True)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU, even on a "T4 x2" machine

GPU_NAME = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.splitlines()[0].strip()
GPU_TAG = GPU_NAME.lower().replace("nvidia", "").replace("tesla", "").strip().replace(" ", "-")
VLLM_VERSION = importlib.metadata.version("vllm")
print(f"GPU: {GPU_NAME} (tag '{GPU_TAG}') · vLLM {VLLM_VERSION} · results -> {RESULTS}")

## 4. Get the benchmark harness

Uses `run_bench.py` and `prompts.jsonl` if you uploaded them (working directory or an attached
Kaggle dataset); otherwise downloads them from GitHub using `GITHUB_USER`.

In [ ]:
def find_or_download(name: str) -> Path:
    candidates = [WORK / name]
    if Path("/kaggle/input").exists():
        candidates += list(Path("/kaggle/input").rglob(name))
    for path in candidates:
        if path.exists():
            print(f"using {path}")
            return path
    if "<" in GITHUB_USER:
        raise SystemExit("Set GITHUB_USER in step 3, or upload run_bench.py and prompts.jsonl")
    dest = WORK / name
    urllib.request.urlretrieve(f"{REPO_RAW}/bench/{name}", dest)
    print(f"downloaded {dest}")
    return dest


RUN_BENCH = find_or_download("run_bench.py")
PROMPTS = find_or_download("prompts.jsonl")

## 5. Helpers: start / stop the vLLM server, run the benchmark

In [ ]:
def start_server(model: str, *extra: str, log_name: str) -> subprocess.Popen:
    """Start `vllm serve` in the background and wait until /v1/models answers."""
    log_path = WORK / f"{log_name}.log"
    cmd = ["vllm", "serve", model, "--port", str(PORT), "--served-model-name", SERVED_NAME,
           "--dtype", "half",                  # the T4 has no bf16 support
           "--max-model-len", "4096",          # same context size as the laptop runs
           "--gpu-memory-utilization", "0.90",
           "--enable-prompt-tokens-details",   # report cached_tokens per request
           *extra]
    print(" ".join(cmd))
    with open(log_path, "w") as log:  # the child keeps its own handle to the file
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    deadline = time.time() + 900
    while time.time() < deadline:
        if proc.poll() is not None:
            raise RuntimeError(f"vllm exited early; see {log_path}")
        try:
            urllib.request.urlopen(f"{BASE_URL}/models", timeout=2)
            print("server ready")
            return proc
        except OSError:
            time.sleep(5)
    proc.kill()
    raise RuntimeError(f"vllm not ready after 15 min; see {log_path}")


def stop_server(proc: subprocess.Popen) -> None:
    proc.terminate()
    try:
        proc.wait(timeout=60)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait()
    time.sleep(10)  # give the GPU memory time to be released


def bench(label: str, experiment: str, *args: str, server_config: dict) -> None:
    cmd = [sys.executable, str(RUN_BENCH), "--base-url", BASE_URL, "--model", SERVED_NAME,
           "--label", label, "--experiment", experiment, "--prompts", str(PROMPTS),
           "--server-config", json.dumps(server_config),
           "--out", str(RESULTS / f"{label}.json"), *args]
    subprocess.run(cmd, check=True)


def server_config(model: str, **kw) -> dict:
    return {"engine": "vllm", "version": VLLM_VERSION, "gpu": GPU_NAME, "model": model,
            "dtype": "float16", "max_model_len": 4096, **kw}

## 6. Experiment 1: concurrency sweep (FP16)

The same 40 prompts and 256 output tokens per request as the laptop baseline, at 1 → 32
concurrent requests. Prefix caching is left on (vLLM's default); the harness tags every request
so only the chat-template prefix can be reused.

In [ ]:
proc = start_server(MODEL, log_name="vllm-fp16")
try:
    bench(f"vllm-{GPU_TAG}-fp16", "load_vllm", "--concurrency", "1,2,4,8,16,32",
          server_config=server_config(MODEL, prefix_caching=True))
finally:
    stop_server(proc)

## 7. Experiment 2: prefix caching on vs. off

Every request shares one ~1.45k-token system prompt and asks a different short question
(12 questions × 4 repeats, 64 output tokens). With prefix caching the shared prefix is computed
once and reused, so time to first token should drop sharply. The server restarts between runs.

In [ ]:
for enabled in (True, False):
    flag = "--enable-prefix-caching" if enabled else "--no-enable-prefix-caching"
    state = "on" if enabled else "off"
    proc = start_server(MODEL, flag, log_name=f"vllm-prefix-{state}")
    try:
        bench(f"vllm-{GPU_TAG}-prefix-{state}", "load_prefix_caching",
              "--prompt-set", "shared_prefix", "--repeat", "4", "--max-tokens", "64",
              "--concurrency", "1,4,16",
              server_config=server_config(MODEL, prefix_caching=enabled))
    finally:
        stop_server(proc)

## 8. (Optional) Experiment 3: AWQ 4-bit model

Qwen's official AWQ checkpoint of the same model, same sweep as Experiment 1: the GPU-side
counterpart to the laptop's GGUF quantization study. Skip this cell if you are short on GPU time.

In [ ]:
proc = start_server(AWQ_MODEL, log_name="vllm-awq")
try:
    bench(f"vllm-{GPU_TAG}-awq", "load_vllm", "--concurrency", "1,2,4,8,16,32",
          server_config=server_config(AWQ_MODEL, quantization="awq", prefix_caching=True))
finally:
    stop_server(proc)

## 9. Summary

In [ ]:
for path in sorted(RESULTS.glob("*.json")):
    run = json.loads(path.read_text())
    n = run["settings"]["num_prompts"]
    print(f"\n{run['label']}  ({run['experiment']}, {n} requests/level)")
    print(f"{'conc':>5} {'tok/s':>8} {'TTFT p50':>9} {'TTFT p95':>9} {'lat p95':>8} "
          f"{'cached':>8} {'errors':>6}")
    for lv in run["levels"]:
        print(f"{lv['concurrency']:>5} {lv['throughput_tps'] or 0:>8.1f} "
              f"{lv['ttft_p50_s'] or 0:>8.2f}s {lv['ttft_p95_s'] or 0:>8.2f}s "
              f"{lv['latency_p95_s'] or 0:>7.1f}s {lv['cached_tokens_total']:>8} {lv['errors']:>6}")

## 10. Download the results

- **Kaggle:** the files are in the notebook's **Output** panel (`results/` and `vllm_results.zip`).
- **Colab:** the cell below also triggers a browser download.

Then copy the JSON files into the repo's `results/` folder and run `uv run python -m serve_llm.plot`.

In [ ]:
import shutil

archive = shutil.make_archive(str(WORK / "vllm_results"), "zip", RESULTS)
print(f"wrote {archive}")
try:
    from google.colab import files  # only exists on Colab

    files.download(archive)
except ImportError:
    pass